# Detecção de Profissões
O objetivo do exercício é criar um classificador capaz de reconhecer a profissão do solicitante com base na requisição recebida. Utiliza-se um conjunto de dados o qual contém o conteúdo da requisição e a profissão do emissor. As categorias consideradas pela classe alvo são: **governament**, **private** e **academic**

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.9 MB/s eta 0:00:00


In [2]:
# importação de bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Importando conjunto de dados ***"ep2-train.csv"***

In [5]:
path_file = "/content/drive/MyDrive/NLP2/ep2-train.csv"
df = pd.read_csv(path_file, encoding='latin1', sep=';')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df.head()

,req_text,profession
0,Sou aluna de doutorado da UFPR e pesquiso sobr...,academic
1,Gostaria de consultar a disponibilidade de cÃ³...,academic
2,"Prezados, bom dia. Procurei no site do Serpro ...",government
3,Solicito o nÃºmero de matrÃ­cula do SIAP do do...,academic
4,A Lei nÂº 12772/2012 regulamenta a carreira de...,academic


In [7]:
df['profession'].value_counts()

,count
profession,
government,18782
academic,14593
private,10303


Como é possível visualizar, ***"ep2-train.csv"*** é um conjunto desbalanceado, contendo majoritariamente requisições de pessoas do **governo**. Logo, a acurácia não é uma métrica confiável para avaliar o desempenho do classificador.

In [8]:
df.isnull().sum()

,0
req_text,0
profession,0


In [9]:
df.duplicated().sum()

np.int64(6898)

O conjunto possui algumas instâncias **repetidas**, mas não há dados **faltantes**. Pode haver requisições iguais com profissões distintas, porém isso será mantido, visto que não como decidir qual é a profissão correta e uma exclusão simples poderia acarretar em uma perda **significativa** de dados. Logo, será retirado apenas as instâncias repetidas considerado todas as **colunas**.

In [10]:
df = df.drop_duplicates() # retira instâncias duplicadas

In [11]:
df.duplicated().sum()

np.int64(0)

Será aplicado uma transformação **Label Encoding** sobre ***"profession"***, por se tratar de uma coluna categórica.

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['profession_encoded'] = le.fit_transform(df['profession']) #transformação númerica simples

In [13]:
df.head()

,req_text,profession,profession_encoded
0,Sou aluna de doutorado da UFPR e pesquiso sobr...,academic,0
1,Gostaria de consultar a disponibilidade de cÃ³...,academic,0
2,"Prezados, bom dia. Procurei no site do Serpro ...",government,1
3,Solicito o nÃºmero de matrÃ­cula do SIAP do do...,academic,0
4,A Lei nÂº 12772/2012 regulamenta a carreira de...,academic,0


## Modelagem da Representação Textual

Com base em experiências passadas, será apenas explorado o impacto da ***representação textual***, pois se viu que o algoritmo utilizado pouco influencia o desempenho. O desempenho é determinado **majoritamente** pela representação utilizada.

In [14]:
#Bag-of-words
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
X = vect.fit_transform(df['req_text']) # BoW com frequência normalizada

In [15]:
X.shape

(36780, 66062)

Foi gerado um Bag-of-Words com um vocabulário de 67.761 palavras (tokens).

Como é um problema desbalanceado e se deseja verificar o desempenho geral, será utilizado a métrica ***F1-Score Weighted***.

In [18]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42)
results = cross_validate(
    estimator=model,
    X = X,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score=False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.595218,0.590339
1,0.590638,0.585483
2,0.583020,0.577503
3,0.577659,0.572785
4,0.587198,0.583119


Como é possível perceber, o uso do BoW não gerou resultados espetáculares. Agora, iremos explorar BoW com TF-IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer()
X_tf = tf_idf.fit_transform(df['req_text']) #BoW TF-IDF

In [20]:
X_tf.shape

(36780, 66062)

BoW TF-IDF gerou a mesma quantidade de atributos.

In [ ]:
results = cross_validate(
    estimator=model,
    X = X_tf,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.624008,0.619461
1,0.623141,0.617982
2,0.608978,0.603795
3,0.610781,0.606055
4,0.619782,0.615032


Fazer uso do **TF-IDF** melhorou relativamente o desempenho do classificador. Com base nisso, iremos explorar o uso de n-gramas com TF-IDF. Iremos testar os seguintes intervalos de n_gramas:
- (1, 2): unigramas, bigramas
- (1, 3): unigramas, bigramas e trigramas

In [21]:
tf_gr1 = TfidfVectorizer(ngram_range=(1,2)) # unigramas e bigramas
X_gr1 = tf_gr1.fit_transform(df['req_text'])
X_gr1.shape

(36780, 712637)

In [ ]:
results = cross_validate(
    estimator=model,
    X = X_gr1,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.641494,0.636543
1,0.641328,0.636305
2,0.633447,0.628360
3,0.634427,0.629099
4,0.638863,0.633699


In [22]:
tf_gr2 = TfidfVectorizer(ngram_range=(1,3)) # Unigramas, bigramas e trigramas
X_gr2 = tf_gr2.fit_transform(df['req_text'])
X_gr2.shape

(36780, 2073210)

In [ ]:
results = cross_validate(
    estimator=model,
    X = X_gr2,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

Houve uma pequena melhora no desempenho ao usar ***n-gramas***. Entretanto, ambos os intervalos testados demonstraram a mesma capacidade preditiva. Logo, será adotado **Unigramas+Bigramas** por conta da dimensionalidade, o qual é menor. Devido ao fenômeno da **maldição da dimensionalidade**, será aplicado uma redução de dimensionalidade.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ("select", SelectKBest(score_func=chi2)),
    ("clf", LogisticRegression(random_state=42))
])

In [ ]:
param_grid = {
    "select__k": [250000, 350000, 450000, 550000, 650000, 700000] #grade de busca
}

In [ ]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1_weighted",
    n_jobs=-1
)

In [ ]:
grid.fit(X_gr1, df['profession_encoded'])

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('select',
                                        SelectKBest(score_func=<function chi2 at 0x7847ddbac4a0>)),
                                       ('clf',
                                        LogisticRegression(random_state=42))]),
             n_jobs=-1,
             param_grid={'select__k': [250000, 350000, 450000, 550000, 650000,
                                       700000]},
             scoring='f1_weighted')

In [ ]:
grid.best_params_

{'select__k': 650000}

In [ ]:
grid.best_score_

np.float64(0.6940646075502215)

Conforme o observado, conseguiu-se manter o desempenho original reduzindo o número de features, diminuindo os efeitos da maldição da dimensionalidade. Logo, para a construção do classificador, será usado ***K=650.000***.

Devido aos resultados obtidos durante o EP anterior, será utilizado o algoritmo **Multinomial Naive Bayes**.

In [25]:

selector = SelectKBest(score_func=chi2, k=650000) # redução de dimensionalidade
X_new = selector.fit_transform(X_gr1, df['profession_encoded'])

In [ ]:
from sklearn.naive_bayes import MultinomialNB
model_nb = MultinomialNB()

In [ ]:
results = cross_validate(
    estimator=model_nb,
    X = X_new,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

,test_f1_weighted,test_f1_macro
0,0.515480,0.463767
1,0.519868,0.468094
2,0.511056,0.459268
3,0.522035,0.471536
4,0.522578,0.470556


O algoritmo Multinominal Naive Bayes obteve um desempenho pior à Regressão Logística.

In [ ]:
from sklearn.svm import SVC
model_svc = SVC(kernel='linear')
results = cross_validate(
    estimator=model_svc,
    X = X_new,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

Sabendo que há uma boa distinção entre o estilo de escrita das profissões abordadas, talvez haja uma melhora no desempenho do classificador ao usar caracterização autoral. No caso, como há somente a requisição, será extraído características relacionadas ao estilo de escrita.

In [ ]:
def extract_author_style(text): #extraí características linguísticas
    words = text.split()
    return {
        'word_count': len(words),
        'avg_word_len': np.mean([len(w) for w in words]) if words else 0,
        'sentece_count': text.count('.'),
        'comma_count': text.count(','),
        'uppercase_ratio': sum(c.isupper() for c in text) / len(text) if text else 0,
        'unique_ratio': len(set(words)) / len(words) if words else 0
    }

In [ ]:
df_features = df['req_text'].apply(lambda t: pd.Series(extract_author_style(t))) # features linguísticas

In [ ]:
from scipy.sparse import hstack

X_combined = hstack([X_new, df_features.values]) # combina vocabulário com estilo de escrita

In [ ]:
results = cross_validate( #features com Regressão Logística
    estimator=model,
    X = X_combined,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

In [ ]:
results = cross_validate( #features com Regressão Logística
    estimator=model,
    X = df_features,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

In [ ]:
df_features.head()

Como foi constatado, o uso das ***features linguísticas*** piorou o desempenho do classificador. Logo, a única alternativa para obter melhores resultados é aplicar algoritmos baseados na arquitetura **transformers**.

Entretanto, testaremos antes o uso de ***word embeddings estáticos***. No caso, treinaremos um modelo próprio de **Word2Vec**, pois essa abordagem tende a capturar melhor **nuancias específicas** do problema. Usaremos o algoritmo de treinamento **Skip-Gram**.

In [ ]:
from gensim.models import Word2Vec

sentences = [text.split() for text in df['req_text']]

In [ ]:
w2v_model = Word2Vec(
    sentences,
    vector_size=300, #tamanho embedding
    window=5,
    min_count=2,
    sg=1, # Skig-gram
    workers=4,
    epochs=10
)

In [ ]:
w2v_model.save("word2vec.model") #salva o modelo de embeddings estáticos

In [ ]:
def text_to_vector(text): #transforma a entrada em embeddings
    words = text.split()
    word_vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    if len(word_vecs) == 0:
        return np.zeros(w2v_model.wv.vector_size)
    return np.mean(word_vecs, axis=0) #agregação por média

In [ ]:
X_embeddings = np.array([text_to_vector(t) for t in df['req_text']])

In [ ]:
results = cross_validate(
    estimator=model,
    X = X_embeddings,
    y = df['profession_encoded'],
    cv = 5,
    scoring = ['f1_weighted', 'f1_macro'],
    return_train_score= False
)

In [ ]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df

# Modelo Final

In [26]:
results = cross_validate(
    estimator=model,
    X = X_new,
    y = df['profession_encoded'],
    cv = 10,
    scoring = ['f1_weighted', 'f1_macro', 'accuracy'],
    return_train_score= False
)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

In [27]:
results_df = pd.DataFrame(results)
results_df.drop(['fit_time', 'score_time'], axis=1, inplace=True)
results_df.mean()

,0
test_f1_weighted,0.653261
test_f1_macro,0.648480
test_accuracy,0.653942


In [28]:
model.fit(X_new, df['profession_encoded']) #Treina modelo final

LogisticRegression(random_state=42)

In [29]:
#Pegando o nome das features utilizadas pelo modelo
feature_names = np.array(tf_gr1.get_feature_names_out())

mask = selector.get_support()

selected_features = feature_names[mask]

In [37]:
test_file = "/content/drive/MyDrive/NLP2/teste/ep2-esic2-profession-test-no-labels.csv"
df_test = pd.read_csv(test_file, encoding='latin1', sep=';')
print(df_test.size)
print(df_test)

6000
                                               req_text
0     Boa tarde, excelentíssimo Sr. Ouvidor do Gover...
1     Boa tarde, solicito a numeração do meu DNI, po...
2     Prezado(a), Gostaria de solicitar um histórico...
3     Gostaria de saber se o servidor caso requisita...
4     Caro(a), solicito as seguintes informações: 1....
...                                                 ...
5995  Prezados: Estou fazendo uma pesquisa sobre os ...
5996  Prezado(a) Sr.(a), Solicito cópia integral, em...
5997  Prezados, Gostaria de saber quantos cargos vag...
5998  Gostaria de saber o andamento do oficio de per...
5999  Quanto mais longa a série histórica dos dados ...

[6000 rows x 1 columns]


In [38]:
X_test_tfidf = tf_gr1.transform(df_test['req_text'])

X_test_new = selector.transform(X_test_tfidf)

y_pred = model.predict(X_test_new)

y_pred_labels = le.inverse_transform(y_pred)

df_test['profession'] = y_pred_labels

df_test.to_csv("predictions.csv", index=False)


In [ ]:
!pip install eli5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 3.9 MB/s eta 0:00:00


In [ ]:
import eli5
eli5.show_weights(model, top=20, feature_names=selected_features, target_names=le.classes_) #Interpreta as top 20 features mais importantes

In [ ]:
eli5.show_prediction(model, X_new[0], top=20, feature_names= selected_features, target_names= le.classes_)

In [ ]:
df['req_text'][0]

In [ ]:
df['profession'][0]